In [30]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/ishmaelsears/janus-repo/janus_cloud_api.py
/kaggle/input/datasets/ishmaelsears/janus-repo/README_computer_use.md
/kaggle/input/datasets/ishmaelsears/janus-repo/task_execution_engine.py
/kaggle/input/datasets/ishmaelsears/janus-repo/updated_enhanced_vision.py
/kaggle/input/datasets/ishmaelsears/janus-repo/janus_selfheal.py
/kaggle/input/datasets/ishmaelsears/janus-repo/test_reflection_dpa.py
/kaggle/input/datasets/ishmaelsears/janus-repo/config_avus_70b.json
/kaggle/input/datasets/ishmaelsears/janus-repo/TODO.md
/kaggle/input/datasets/ishmaelsears/janus-repo/face_smile.obj
/kaggle/input/datasets/ishmaelsears/janus-repo/core_updated.py
/kaggle/input/datasets/ishmaelsears/janus-repo/runtime_error.txt
/kaggle/input/datasets/ishmaelsears/janus-repo/jce.rs
/kaggle/input/datasets/ishmaelsears/janus-repo/janus_user_frontend.py
/kaggle/input/datasets/ishmaelsears/janus-repo/desk_janus.db
/kaggle/input/datasets/ishmaelsears/janus-repo/janus_autonomous_real.py
/kaggle/input

In [31]:
import os
import shutil

# Remove the staging directory to free up space
staging_path = '/kaggle/working/upload_staging'
if os.path.exists(staging_path):
    shutil.rmtree(staging_path)
    print("Cleaned up staging area.")

In [32]:
import torch.nn as nn

# 1. Store the TRUE original only if we haven't already
if not hasattr(nn.Linear, '_is_patched'):
    _REAL_ORIG_LINEAR = nn.Linear.forward

    def _device_aware_linear(self, x):
        # Move weights/bias only if necessary to avoid overhead
        if self.weight.device != x.device:
            self.weight.data = self.weight.data.to(x.device)
        if self.bias is not None and self.bias.device != x.device:
            self.bias.data = self.bias.data.to(x.device)
            
        # Use the captured REAL original function
        return _REAL_ORIG_LINEAR(self, x)

    # 2. Apply the patch and mark it
    nn.Linear.forward = _device_aware_linear
    nn.Linear._is_patched = True
    print("[avus] Applied recursion-safe Linear patch.")

In [33]:
# Example of manual sharding in your Avus class
def __init__(self, config):
    super().__init__()
    self.blocks = nn.ModuleList([Block(config) for _ in range(config.n_layers)])
    
    # Manually assign blocks to different GPUs
    for i, block in enumerate(self.blocks):
        device = f'cuda:{0 if i < config.n_layers // 2 else 1}'
        block.to(device)

In [34]:
import torch.nn as nn

# Check if we have already patched nn.Linear to prevent recursion loops
if not hasattr(nn.Linear, '_is_patched'):
    # Capture the REAL original forward method
    _REAL_PYTORCH_LINEAR_FORWARD = nn.Linear.forward

    def _device_aware_linear(self, x):
        # Ensure weights are on the correct device before the forward pass
        if self.weight.device != x.device:
            self.weight.data = self.weight.data.to(x.device)
        if self.bias is not None and self.bias.device != x.device:
            self.bias.data = self.bias.data.to(x.device)
            
        # Call the ACTUAL original PyTorch method
        return _REAL_PYTORCH_LINEAR_FORWARD(self, x)

    # Apply the patch
    nn.Linear.forward = _device_aware_linear
    # Mark it so we don't do this again in the same session
    nn.Linear._is_patched = True
    print("[avus] Successfully applied recursion-safe device-aware patch.")
else:
    print("[avus] nn.Linear already patched; skipping to avoid recursion.")

[avus] nn.Linear already patched; skipping to avoid recursion.


In [35]:
!python /kaggle/input/datasets/ishmaelsears/janus-repo/kaggle_launcher.py

In [36]:
!python /kaggle/input/datasets/ishmaelsears/janus-repo/train_avus_kaggle_fixed.py

In [37]:
!python /kaggle/input/datasets/ishmaelsears/janus-repo/direct_fix.py

Starting direct fix for HBM complex number issue...
=== APPLYING DIRECT FIX FOR COMPLEX NUMBERS ===
✅ Applied fix 1 to HBM core
Traceback (most recent call last):
  File "/kaggle/input/datasets/ishmaelsears/janus-repo/direct_fix.py", line 147, in <module>
    if apply_direct_fix():
       ^^^^^^^^^^^^^^^^^^
  File "/kaggle/input/datasets/ishmaelsears/janus-repo/direct_fix.py", line 93, in apply_direct_fix
    with open(hbm_core_path, 'w') as f:
         ^^^^^^^^^^^^^^^^^^^^^^^^
OSError: [Errno 30] Read-only file system: '/kaggle/input/datasets/ishmaelsears/janus-repo/holographic_brain_memory/core.py'


In [38]:
import os, sys, json, torch

REPO = None
for root, dirs, files in os.walk("/kaggle/input"):
    if "train_avus_kaggle.py" in files:
        REPO = root
        break

if REPO is None:
    raise FileNotFoundError("train_avus_kaggle.py not found.")

print(f"Repo found at: {REPO}")
sys.path.insert(0, REPO)

# Patch nn.Linear and RMSNorm to be device-aware (model parallelism fix)
import torch.nn as nn
_orig_linear_forward = nn.Linear.forward
def _device_aware_linear(self, x):
    if self.weight.device != x.device:
        self.weight = nn.Parameter(self.weight.to(x.device), requires_grad=self.weight.requires_grad)
        if self.bias is not None:
            self.bias = nn.Parameter(self.bias.to(x.device), requires_grad=self.bias.requires_grad)
    return _orig_linear_forward(self, x)
nn.Linear.forward = _device_aware_linear

# Patch avus.py RMSNorm
avus_src = open(os.path.join(REPO, "avus.py")).read()
avus_src = avus_src.replace(
    "        norm = torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)\n        return x * norm * self.weight",
    "        if self.weight.device != x.device:\n            self.weight = torch.nn.Parameter(self.weight.to(x.device), requires_grad=self.weight.requires_grad)\n        norm = torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)\n        return x * norm * self.weight"
)
with open("/kaggle/working/avus.py", "w") as f:
    f.write(avus_src)
sys.path.insert(0, "/kaggle/working")

try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("Kiro")
    kaggle_dir = os.path.expanduser("~/.kaggle")
    os.makedirs(kaggle_dir, exist_ok=True)
    creds = {"username": "ishmaelsears", "key": token}
    with open(os.path.join(kaggle_dir, "kaggle.json"), "w") as f:
        json.dump(creds, f)
    os.chmod(os.path.join(kaggle_dir, "kaggle.json"), 0o600)
    print("Kaggle API ready")
except Exception as e:
    print(f"No 'Kiro' secret ({e}) — auto-push disabled")

# Patch focal loss device fix
script = open(os.path.join(REPO, "train_avus_kaggle.py")).read()
script = script.replace("KAGGLE_MODE           = False", "KAGGLE_MODE           = True")
script = script.replace(
    "        ce   = F.cross_entropy(logits, targets, ignore_index=-1, reduction=\"none\")",
    "        targets = targets.to(logits.device)\n        ce   = F.cross_entropy(logits, targets, ignore_index=-1, reduction=\"none\")"
)
exec(script)


Repo found at: /kaggle/input/datasets/ishmaelsears/janus-repo
Kaggle API ready

[KaggleMode] Applying T4 x2 optimizations...
[KaggleMode] Config: dim=1920 layers=20 ~0.88B params
[KaggleMode] Batch=1 GradAccum=16 EffectiveBatch=16 SeqLen=256
[KaggleMode] GradScaler disabled
[KaggleMode] GPU-resident optimizer — CUDA unified memory paging enabled
[KaggleMode] Optimizer states will stay on GPU (fast steps)
[KaggleMode] DataParallel disabled
[KaggleMode] AvusBlock device-aware forward enabled
[KaggleMode] ModelParallel enabled across 2 GPUs
[KaggleMode] All optimizations applied. Ready to train.

[setup] Janus repo found at /kaggle/working
[setup] Skill curriculum loaded
[setup] HBM modules loaded
[setup] PyTorch 2.10.0+cu128 | CUDA: True
[setup] GPU 0: Tesla T4 | 15.6 GB VRAM
[setup] GPU 1: Tesla T4 | 15.6 GB VRAM

TRAINING AVUS-1B
[avus] Config from /kaggle/working/config_avus_1b.json
[avus] dim=1920 layers=20 heads=16 kv=8 ffn=5120 seq=256
[KaggleMode] ModelParallel: GPU0 layers 0-9 | 

RecursionError: maximum recursion depth exceeded

In [ ]:
from IPython.display import FileLink

# Generate download links for your outputs
display(FileLink(r'avus_1b_weights.pt'))
display(FileLink(r'skill_chart.png'))